# Apache Spark: Architecture, Optimization, and Best Practices Day 1 Lecture

Note that I already use Spark and PySpark in my day-to-day for my work so notes in this section will likely be light.

The driver has many settings but two are particularly important:
- spark.driver.memory: For complex jobs or jobs that use dataframe.collect(), you may need to bump this higher or else you'll experience and OOM. This config has a default value for memory, but I may wanna double check this in case things have changed.
- spark.driver.memoryOverheadFactor: What fraction the driver needs for non-heap related memory (memory needed for jvm), usually 10%, might need to be higher for complext jobs.

At a high-level, there are 3 components to spark:
- The plan: the transformation you describe in your code
- The driver
- The executors

Executor settings that are important:
- spark.executor.memory: determines how much memory each executor gets. A low number here may cause Spark to spill to disk, which will cause your job to run much slower. I believe databricks handles this for us but I'll need to double check. Default he said is 1 or 2, but you can bump it up to 16.
- spark.executor.cores: how many tasks can happen on each machine. Default is 4, shouldn't go higher than 6. I'm pretty sure databricks handles this too, but will have to double check. He also stated he doesn't really touch this.
- spark.executor.memoryOverheadFactor: what % of memory should an executor use for non-heap related tasks. Usually 10%. For jobs with lots of UDFs and complexity, you may need to bump this up.

There are 3 types of joins in spark:
- Shuffle sort-merge join
    - default join
- broadcast join
    - theres a setting that will trigger a broadcast join automatically at a certain memory size
- Bucket joins
    - a join without shuffle

Shuffle partitions and parallelism are linked. Default partitions is 200.

He said shuffle honestly ain't that bad. But when you're working with high volume data (> 10 TB), then that's when you probably won't be able to shuffle.

Bucket joins are only good if you have multiple joins/aggregations downstream.

Be careful using Spark for API calls. API calls are done on the driver. General recommendation is not to do this. Try pulling api in an earlier process and save out somewhere and then have spark read from there.

Spark output datasets should almost always be partitioned on "date". Whether that's execution date of the data or transaction date for a fact table. Depends on how its queried.

## Lab

VERY IMPORTANT: in order to be able to optimize your queries, you're gonna have to get familiar with reading spark plans. You can check out the plans by doing a `df.explain()`

In this lab, he showed us how to important it was when saving files out, to `sortWithinPartitions`. This will reduce the size of your dataset significantly due to run length encoding.

The cool thing was that he was querying iceberg metadata files to get byte size of files.

# Apache Spark: Managing Spark Jobs and Notebooks Day 2 Lecture

Spark server vs spark notebooks
- Spark server (how airbnb does it)
    - gotta use the cli to submit jobs
- notebook (how netflix does it)

This is interesting because Databricks really abstracts a lot of this away from users.

Caching to get help speed up operations if an upstream temp view is used multiple times. However, if you're caching a big dataframe, it's often a good idea to save that data as a table. We actually do this in my work.

Shuffle partitions
- Default is 200
- Aim for 100 MBs per partition to get the right sized output datasets

To figure this out, you can do back of the envelope math. Might be good to test 1 higher and 1 lower too to see performance on pipeline.

## Lab

He stated how you should only be using cache() when caching tables. That's because it persists within memory. Persist does so on disk. And at that point, you might as well save it as a staging table.

He also walks thru a bucket join example.

# Unit Testing Spark Jobs: Importance, Challenges, and Leadership Perspectives Lecture

Where can you catch quality bugs?
- In development (best case)
- In production, but not showing up in tables (still good)
    - Think about catching something during auditing step of write audit publish
- In production, in production tables (terrible and destorys trust)

How do you catch bugs in dev?
- Unit tests and integration tets of your pipelines
    - this is very important when you rely on other packages

How do you catch bugs in production?
- Use the write-audit-publish

He discussed a lot on how important it is to think of data engineering like software engineering. It's so important to build things the right way with the best possible quality checks. This is something I absolutely agree with.

Silent failures are your enemy. Loud failures are your friend.

## Lab

In this lab, we walk thru an example of using unit tests to test code.